In [ ]:
# Canonical analysis definitions (single source of truth: analysis/_canonical_defs.py).
# Importing these overrides any local fallback definitions later in the notebook.
# Edits to filters / score-mapping / bucket-rule belong in _canonical_defs.py, not here.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("analysis").resolve()))
from _canonical_defs import (
    ANCHOR, L2_CELLS, AXIS_CELLS, VERDICT_SCORE,
    candidate_score, sign,
    any_tier_disagreement, boundary_flip, strict_reversal,
    same_side_magnitude_shift, strong_to_weak_shift,
    assign_bucket,
)


# 10d — Pre-submission robustness checks

Six analyses queued by the verifier (Adversarial Review v2, 2026-05-02 → 2026-05-03). All are pure recomputes on existing data — no new judge calls. Estimated total runtime: ~30–60 min on `verdict_scores_wide.csv` and friends.

**Sequence (in priority-of-leverage order):**

| # | Analysis | What it disambiguates | Output | Maps to |
|---|---|---|---|---|
| 1 | **Variance-matched CoT subsample** | Whether F1's larger Style Control adjustment is mechanism-driven (verbalisation channel) or statistical artefact (F1 has more variance to fit) | `results/variance_matched_cot.csv` | A-008 / N-001 |
| 2 | **Bootstrap-anchor-stability** | Whether anchor C1F1D1 win-rate is bootstrap-distinguishable from each other cell (parallels CAP A-CAP-007's 50.7%) | `results/bootstrap_anchor_stability.csv` | New empirical claim — paper-level symmetry with CAP |
| 3 | **Bootstrap seed-sensitivity** | Whether bucket-distribution CIs are stable across seeds (verifier's single-seed concern) | `results/bootstrap_seed_sensitivity.csv` | A-001 robustness |
| 4 | **Anchor-cell bootstrap CI** | Whether Arena-Hard's published 48.6% (gpt-4.1-mini under GPT-4.1 with style control) falls inside our anchor-cell CI | `results/anchor_cell_ci.json` | A-007 / D-001 |
| 5 | **Invalid-verdict rate audit** | Per-cell rate of malformed/refused/unparseable verdicts; confound source | `results/invalid_verdict_audit.csv` | Methods/limitations |
| 6 | **Jaccard chance-baseline computation in-cell** | Adds Obs/Chance ratios to nb 10's Jaccard table (currently in `paper_claims.md` A-013 only) | `results/jaccard_with_baselines.csv` | A-013 |

**Inputs:** `verdict_scores_wide.csv`, `style_features.csv`, `controlled_winrates.csv`, raw `data/full_factorial/*.jsonl` and `data/L1_swap/C1F1D1.jsonl`.

**Outputs:** six new CSV/JSON files under `results/`, plus a final summary cell with paper-relevant headline numbers.

In [1]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from sklearn.linear_model import LinearRegression

results_dir = Path('results')
wide = pd.read_csv(results_dir / 'verdict_scores_wide.csv', index_col='uid')
style_df = pd.read_csv(results_dir / 'style_features.csv').set_index('uid')
control_df = pd.read_csv(results_dir / 'controlled_winrates.csv')

L2_CELLS = ['C1F1D1', 'C1F1D2', 'C1F2D1', 'C1F2D2', 'C2F1D1', 'C2F1D2', 'C2F2D1', 'C2F2D2']
ANCHOR = 'C1F1D1'
F1_CELLS = ['C1F1D1', 'C1F1D2', 'C2F1D1', 'C2F1D2']
F2_CELLS = ['C1F2D1', 'C1F2D2', 'C2F2D1', 'C2F2D2']
ALL_CELLS = list(wide.columns)

print(f'Loaded: wide={wide.shape}, style_df={style_df.shape}, control_df={control_df.shape}')
print(f'Cells: {ALL_CELLS}')

Loaded: wide=(500, 10), style_df=(500, 4), control_df=(10, 9)
Cells: ['C1F1D1', 'C1F1D2', 'C1F2D1', 'C1F2D2', 'C2F1D1', 'C2F1D2', 'C2F2D1', 'C2F2D2', 'L1_claude', 'L1_gemini']


## 1. Variance-matched CoT subsample (HIGHEST LEVERAGE)

**Concern (A-008 / N-001):** F1 cells lose 3–4pp under Style Control; F2 cells lose 0.1–0.9pp. Two interpretations:
- **(a) Verbalisation-channel mechanism:** reasoning verbalises style cues into substantive justifications, amplifying influence.
- **(b) Statistical artefact:** F2 has lower verdict variance (more ties, mean_score ~0), so the regression intercept mechanically moves less even if underlying style-influence is similar.

**Test:** subsample F1 prompts to match F2's verdict-score variance, re-run Style Control on the matched subsample, see if the 3–4pp drop survives.
- **Survives matched variance →** mechanism (a) verbalisation channel
- **Drops to F2-level (~0.1–0.9pp) →** mechanism (b) variance artefact
- **Intermediate →** both contribute; report honestly

**Method:** for each F1 cell, greedy-trim prompts (largest |mean_score| first) until variance matches the average F2 cell. Re-run the Style Control regression on the trimmed subset. Report adjusted win-rate at zero style differential.

In [2]:
# Compute target variance (mean of F2 cells' var(mean_score))
f2_var = np.mean([wide[c].dropna().var() for c in F2_CELLS])
print(f'Target variance (mean of F2 cells\' verdict variance): {f2_var:.4f}')
for c in F1_CELLS + F2_CELLS:
    print(f'  {c}: var(mean_score) = {wide[c].dropna().var():.4f}, mean = {wide[c].dropna().mean():+.3f}')

def trim_to_variance(scores, target_var, tol=0.005):
    """Greedy-trim largest |score| prompts until variance ≤ target_var."""
    s = scores.dropna().copy()
    while s.var() > target_var + tol and len(s) > 50:
        s = s.drop(s.abs().idxmax())
    return s.index

def style_controlled_winrate(uids, cell_id):
    """Run Style Control regression on a UID subset and return adjusted win-rate."""
    sub = wide.loc[wide.index.intersection(uids), cell_id].dropna()
    if len(sub) < 30:
        return None
    df = pd.DataFrame({'mean_score': sub})
    df = df.merge(style_df[['length_diff', 'md_diff']], left_index=True, right_index=True)
    df['win_equiv'] = (df['mean_score'] + 2) / 4
    X = df[['length_diff', 'md_diff']].values
    y = df['win_equiv'].values
    reg = LinearRegression().fit(X, y)
    raw = df['win_equiv'].mean() * 100
    adj = reg.intercept_ * 100
    return {'n': len(df), 'raw_pct': raw, 'adjusted_pct': adj, 'delta_pp': raw - adj}

rows = []
for cell in F1_CELLS:
    matched_uids = trim_to_variance(wide[cell], f2_var)
    full = style_controlled_winrate(wide[cell].dropna().index, cell)
    matched = style_controlled_winrate(matched_uids, cell)
    rows.append({
        'cell': cell,
        'full_n': full['n'],
        'full_var': round(wide[cell].dropna().var(), 4),
        'full_delta_pp': round(full['delta_pp'], 2),
        'matched_n': matched['n'],
        'matched_var': round(wide[cell].loc[matched_uids].var(), 4),
        'matched_delta_pp': round(matched['delta_pp'], 2),
        'delta_change': round(full['delta_pp'] - matched['delta_pp'], 2),
    })

vmc_df = pd.DataFrame(rows)
print('\nVariance-matched CoT subsample — Style Control adjustment per F1 cell:')
print(vmc_df.to_string(index=False))
vmc_df.to_csv(results_dir / 'variance_matched_cot.csv', index=False)

f2_avg_delta = control_df[control_df.cell_id.isin(F2_CELLS)]['delta_pp'].mean()
print(f'\nReference F2 average delta_pp: {f2_avg_delta:.2f}')
print(f'Mean F1 full delta: {vmc_df.full_delta_pp.mean():.2f}pp')
print(f'Mean F1 matched delta: {vmc_df.matched_delta_pp.mean():.2f}pp')
print('\nInterpretation:')
if vmc_df.matched_delta_pp.mean() < f2_avg_delta + 0.5:
    print('  → Matched F1 ≈ F2 → variance artefact (mechanism b) DOMINATES')
elif vmc_df.matched_delta_pp.mean() > vmc_df.full_delta_pp.mean() - 0.5:
    print('  → Matched F1 ≈ Full F1 → verbalisation-channel (mechanism a) SURVIVES variance match')
else:
    print('  → Intermediate: both mechanisms contribute, report honestly')
print('Saved: results/variance_matched_cot.csv')

Target variance (mean of F2 cells' verdict variance): 0.6955
  C1F1D1: var(mean_score) = 1.1828, mean = +0.127
  C1F1D2: var(mean_score) = 1.3054, mean = +0.165
  C2F1D1: var(mean_score) = 1.2817, mean = +0.126
  C2F1D2: var(mean_score) = 1.2902, mean = +0.141
  C1F2D1: var(mean_score) = 0.7465, mean = -0.006
  C1F2D2: var(mean_score) = 0.7960, mean = -0.009
  C2F2D1: var(mean_score) = 0.5974, mean = -0.017
  C2F2D2: var(mean_score) = 0.6422, mean = -0.010

Variance-matched CoT subsample — Style Control adjustment per F1 cell:
  cell  full_n  full_var  full_delta_pp  matched_n  matched_var  matched_delta_pp  delta_change
C1F1D1     496    1.1828           2.99        414       0.6967              2.93          0.06
C1F1D2     499    1.3054           2.51        385       0.6989              3.36         -0.85
C2F1D1     500    1.2817           3.53        411       0.6975              4.19         -0.66
C2F1D2     500    1.2902           4.28        409       0.6975              3.29  

## 2. Bootstrap-anchor-stability

**Parallels CAP A-CAP-007** (anchor wins aggregate accuracy in only 50.7% of 1000 resamples). For Paradigm 2: bootstrap 500 prompts × 1000 iter; for each resample, check whether the anchor C1F1D1 win-rate is bootstrap-distinguishable from each other cell.

**Reading:**
- If anchor wins each comparison in <60% of resamples → anchor is statistically tied with the other cell → H1 (macro-stable / micro-churning) lands symmetrically across paradigms.
- If anchor wins in >75% of resamples → anchor is bootstrap-distinguishable → reframe needed.

**Output:** for each (anchor, other-cell) pair, the % of bootstrap resamples where anchor's win-rate exceeds the other cell's.

In [3]:
def winrate_per_cell(sub_wide):
    """Compute (mean_score + 2) / 4 win-equivalent per cell, averaged over prompts."""
    return ((sub_wide + 2) / 4).mean()

def bootstrap_anchor_stability(wide, anchor=ANCHOR, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    other_cells = [c for c in wide.columns if c != anchor]
    wins = {c: 0 for c in other_cells}
    deltas = {c: [] for c in other_cells}
    n = len(wide)
    for _ in range(n_iter):
        sample_idx = rng.choice(wide.index, size=n, replace=True)
        sub = wide.loc[sample_idx]
        wr = winrate_per_cell(sub)
        for c in other_cells:
            if pd.notna(wr[anchor]) and pd.notna(wr[c]):
                if wr[anchor] > wr[c]:
                    wins[c] += 1
                deltas[c].append((wr[anchor] - wr[c]) * 100)
    rows = []
    for c in other_cells:
        d = np.array(deltas[c])
        rows.append({
            'comparison': f'{anchor} vs {c}',
            'anchor_wins_pct': 100 * wins[c] / n_iter,
            'mean_delta_pp': d.mean(),
            'ci_lo_pp': np.percentile(d, 2.5),
            'ci_hi_pp': np.percentile(d, 97.5),
            'distinguishable': not (np.percentile(d, 2.5) < 0 < np.percentile(d, 97.5)),
        })
    return pd.DataFrame(rows)

stability = bootstrap_anchor_stability(wide, n_iter=1000, seed=42)
print('Bootstrap-anchor-stability (1000 iter, seed=42):')
print(stability.round(2).to_string(index=False))
stability.to_csv(results_dir / 'bootstrap_anchor_stability.csv', index=False)

tied = (stability['anchor_wins_pct'].between(40, 60)).sum()
distinguishable = stability['distinguishable'].sum()
print(f'\nSummary: {tied} of {len(stability)} comparisons are statistically tied (anchor wins 40–60% of resamples)')
print(f'         {distinguishable} of {len(stability)} are bootstrap-distinguishable (CI excludes 0)')
print(f'\nCAP A-CAP-007 reference: anchor wins aggregate accuracy in 50.7% of 1000 resamples')
print('Saved: results/bootstrap_anchor_stability.csv')

Bootstrap-anchor-stability (1000 iter, seed=42):
         comparison  anchor_wins_pct  mean_delta_pp  ci_lo_pp  ci_hi_pp  distinguishable
   C1F1D1 vs C1F1D2             10.5          -0.94     -2.31      0.51            False
   C1F1D1 vs C1F2D1            100.0           3.33      1.68      4.97             True
   C1F1D1 vs C1F2D2            100.0           3.42      1.78      5.10             True
   C1F1D1 vs C2F1D1             54.5           0.07     -1.69      1.73            False
   C1F1D1 vs C2F1D2             36.2          -0.30     -1.97      1.38            False
   C1F1D1 vs C2F2D1            100.0           3.61      1.84      5.33             True
   C1F1D1 vs C2F2D2            100.0           3.43      1.71      5.14             True
C1F1D1 vs L1_claude            100.0           4.01      2.02      5.86             True
C1F1D1 vs L1_gemini             43.0          -0.16     -2.31      2.22            False

Summary: 2 of 9 comparisons are statistically tied (anchor w

## 3. Bootstrap seed-sensitivity check

**Verifier concern:** the bucket-distribution CIs in A-001 use a single seed (42). Different seeds could give meaningfully different intervals. Re-run the bootstrap with 5 different seeds; report CI range across seeds.

**Pass threshold:** if CI bounds vary by <2pp across seeds, the original A-001 CI is robust.

In [4]:
def assign_bucket(row):
    signs = np.sign(row.dropna())
    if len(signs) < 4:
        return 'B3_unresolved'
    pos = (signs > 0).sum()
    neg = (signs < 0).sum()
    zero = (signs == 0).sum()
    n = len(signs)
    if pos == n or neg == n or zero == n:
        return 'B1_stable'
    majority = max(pos, neg, zero) / n
    if majority < 0.6 or zero > n * 0.4:
        return 'B3_unresolved'
    return 'B2_sensitive'

def bootstrap_buckets(l2_wide, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(l2_wide)
    proportions = []
    for _ in range(n_iter):
        sampled = l2_wide.sample(n=n, replace=True, random_state=rng.integers(0, 1_000_000))
        b = sampled.apply(assign_bucket, axis=1)
        props = b.value_counts(normalize=True).to_dict()
        proportions.append({k: props.get(k, 0) for k in ['B1_stable', 'B2_sensitive', 'B3_unresolved']})
    return pd.DataFrame(proportions)

l2_wide = wide[L2_CELLS]
seeds = [42, 1, 7, 123, 2026]
rows = []
for s in seeds:
    boot = bootstrap_buckets(l2_wide, n_iter=1000, seed=s)
    for col in ['B1_stable', 'B2_sensitive', 'B3_unresolved']:
        rows.append({
            'seed': s,
            'bucket': col,
            'mean': round(boot[col].mean() * 100, 2),
            'ci_lo': round(np.percentile(boot[col] * 100, 2.5), 2),
            'ci_hi': round(np.percentile(boot[col] * 100, 97.5), 2),
        })

seed_df = pd.DataFrame(rows)
print('Bootstrap seed-sensitivity (1000 iter per seed × 5 seeds):')
print(seed_df.to_string(index=False))
seed_df.to_csv(results_dir / 'bootstrap_seed_sensitivity.csv', index=False)

print('\nCI-bound spread across seeds (per bucket):')
for bucket in ['B1_stable', 'B2_sensitive', 'B3_unresolved']:
    sub = seed_df[seed_df.bucket == bucket]
    lo_spread = sub.ci_lo.max() - sub.ci_lo.min()
    hi_spread = sub.ci_hi.max() - sub.ci_hi.min()
    print(f'  {bucket}: CI lo spread {lo_spread:.2f}pp; CI hi spread {hi_spread:.2f}pp — {"ROBUST" if max(lo_spread, hi_spread) < 2 else "SENSITIVE"}')
print('Saved: results/bootstrap_seed_sensitivity.csv')

Bootstrap seed-sensitivity (1000 iter per seed × 5 seeds):
 seed        bucket  mean  ci_lo  ci_hi
   42     B1_stable 30.06  26.00   34.4
   42  B2_sensitive 25.65  21.80   29.6
   42 B3_unresolved 44.29  40.20   48.4
    1     B1_stable 30.09  26.00   34.2
    1  B2_sensitive 25.63  22.00   29.2
    1 B3_unresolved 44.28  39.99   48.6
    7     B1_stable 29.99  26.00   34.2
    7  B2_sensitive 25.62  22.00   29.6
    7 B3_unresolved 44.39  40.20   48.6
  123     B1_stable 30.02  26.20   34.4
  123  B2_sensitive 25.67  22.00   29.6
  123 B3_unresolved 44.32  40.20   48.8
 2026     B1_stable 30.12  26.00   34.0
 2026  B2_sensitive 25.53  21.40   29.0
 2026 B3_unresolved 44.35  39.99   49.0

CI-bound spread across seeds (per bucket):
  B1_stable: CI lo spread 0.20pp; CI hi spread 0.40pp — ROBUST
  B2_sensitive: CI lo spread 0.60pp; CI hi spread 0.60pp — ROBUST
  B3_unresolved: CI lo spread 0.21pp; CI hi spread 0.60pp — ROBUST
Saved: results/bootstrap_seed_sensitivity.csv


## 4. Anchor-cell bootstrap CI for Style Control reproduction

**Question (A-007 / D-001):** does Arena-Hard's published 48.6% (gpt-4.1-mini under GPT-4.1 judge with style control) fall inside our anchor-cell bootstrap CI on the 50.2% adjusted win-rate?

**Method:** bootstrap-resample the 496 anchor prompts × 1000 iter; for each resample, run the Style Control regression and compute adjusted win-rate. Report 2.5/97.5 percentile.

In [5]:
anchor_uids = wide[ANCHOR].dropna().index
anchor_data = pd.DataFrame({'mean_score': wide.loc[anchor_uids, ANCHOR]})
anchor_data = anchor_data.merge(style_df[['length_diff', 'md_diff']], left_index=True, right_index=True)
anchor_data['win_equiv'] = (anchor_data['mean_score'] + 2) / 4

rng = np.random.default_rng(42)
boot_adjusted = []
for _ in range(1000):
    sample = anchor_data.sample(n=len(anchor_data), replace=True, random_state=rng.integers(0, 1_000_000))
    X = sample[['length_diff', 'md_diff']].values
    y = sample['win_equiv'].values
    reg = LinearRegression().fit(X, y)
    boot_adjusted.append(reg.intercept_ * 100)

boot_adjusted = np.array(boot_adjusted)
ci_lo, ci_hi = np.percentile(boot_adjusted, [2.5, 97.5])
anchor_ci = {
    'n': int(len(anchor_data)),
    'point_estimate_pct': float(50.18),
    'bootstrap_mean_pct': float(boot_adjusted.mean()),
    'ci_lo_pct': float(ci_lo),
    'ci_hi_pct': float(ci_hi),
    'arena_hard_published_pct': 48.6,
    'published_inside_ci': bool(ci_lo <= 48.6 <= ci_hi),
    'gap_to_published_pp': float(50.18 - 48.6),
}
print('Anchor-cell Style Control bootstrap CI:')
for k, v in anchor_ci.items():
    print(f'  {k}: {v}')

with open(results_dir / 'anchor_cell_ci.json', 'w') as f:
    json.dump(anchor_ci, f, indent=2)
print('\nSaved: results/anchor_cell_ci.json')

Anchor-cell Style Control bootstrap CI:
  n: 496
  point_estimate_pct: 50.18
  bootstrap_mean_pct: 50.14763248062813
  ci_lo_pct: 46.503723694502305
  ci_hi_pct: 53.686931596110945
  arena_hard_published_pct: 48.6
  published_inside_ci: True
  gap_to_published_pp: 1.5799999999999983

Saved: results/anchor_cell_ci.json


## 5. Invalid-verdict rate audit

**Confound check:** what fraction of judge calls produced malformed / refused / unparseable verdicts? Computed per cell + round. If non-trivial (>5% in any cell), flag in methods.

Valid verdict tags: `A>>B`, `A>B`, `A=B`, `B>A`, `B>>A`. Anything else (None, empty, parse failure, refusal text) counts as invalid.

In [6]:
# [L1_GEMINI_PATCH] This cell conditionally handles L1_claude. After Gemini
# data lands in verdict_scores_wide.csv, the same logic applies to L1_gemini.
# Manual extension may be needed — review the 'L1_claude' references below.
VALID_VERDICTS = {'A>>B', 'A>B', 'A=B', 'B>A', 'B>>A'}
L2_DIR = Path('data/full_factorial')
L1_DIR = Path('data/L1_swap')

def audit_cell(path, cell_id):
    rows = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            for round_name in ['round1', 'round2']:
                v = r.get(round_name, {}).get('verdict')
                rows.append({
                    'cell_id': cell_id,
                    'round': round_name,
                    'uid': r['uid'],
                    'verdict': v,
                    'is_valid': v in VALID_VERDICTS,
                })
    return rows

audit_rows = []
for cell in L2_CELLS:
    audit_rows.extend(audit_cell(L2_DIR / f'{cell}.jsonl', cell))
if (L1_DIR / f'{ANCHOR}.jsonl').exists():
    audit_rows.extend(audit_cell(L1_DIR / f'{ANCHOR}.jsonl', 'L1_claude'))
L1_DIR_GEMINI = Path('data/L1_swap_v2_flash_16k')  # Gemini 2.5 Flash judge; uniform 16k cap
if (L1_DIR_GEMINI / f'{ANCHOR}.jsonl').exists():
    audit_rows.extend(audit_cell(L1_DIR_GEMINI / f'{ANCHOR}.jsonl', 'L1_gemini'))

audit_df = pd.DataFrame(audit_rows)
summary = audit_df.groupby(['cell_id', 'round']).agg(
    total=('uid', 'count'),
    valid=('is_valid', 'sum'),
).reset_index()
summary['invalid'] = summary['total'] - summary['valid']
summary['invalid_pct'] = round(100 * summary['invalid'] / summary['total'], 2)

print('Invalid-verdict rate per cell × round:')
print(summary.to_string(index=False))
summary.to_csv(results_dir / 'invalid_verdict_audit.csv', index=False)

max_invalid = summary['invalid_pct'].max()
max_row = summary.loc[summary['invalid_pct'].idxmax()]
print(f'\nMax invalid rate: {max_invalid}% ({max_row.cell_id} {max_row["round"]})')
print(f'Total invalid across all cells × rounds: {summary["invalid"].sum()} / {summary["total"].sum()} ({100 * summary["invalid"].sum() / summary["total"].sum():.2f}%)')
print('Saved: results/invalid_verdict_audit.csv')

Invalid-verdict rate per cell × round:
  cell_id  round  total  valid  invalid  invalid_pct
   C1F1D1 round1    500    496        4          0.8
   C1F1D1 round2    500    499        1          0.2
   C1F1D2 round1    500    499        1          0.2
   C1F1D2 round2    500    500        0          0.0
   C1F2D1 round1    500    500        0          0.0
   C1F2D1 round2    500    500        0          0.0
   C1F2D2 round1    500    500        0          0.0
   C1F2D2 round2    500    500        0          0.0
   C2F1D1 round1    500    500        0          0.0
   C2F1D1 round2    500    500        0          0.0
   C2F1D2 round1    500    500        0          0.0
   C2F1D2 round2    500    500        0          0.0
   C2F2D1 round1    500    500        0          0.0
   C2F2D1 round2    500    500        0          0.0
   C2F2D2 round1    500    500        0          0.0
   C2F2D2 round2    500    500        0          0.0
L1_claude round1    498    498        0          0.0
L1_clau

## 6. Jaccard chance baselines (in-cell computation)

**Updates A-013** to include Obs/Chance ratios derived from marginal flip rates. Currently nb 06 prints raw Jaccard but not the chance baseline.

Under independence: E[Jaccard(A, B)] = p_A · p_B / (p_A + p_B − p_A · p_B), where p_X is the marginal flip rate of axis X.

In [7]:
# [L1_GEMINI_PATCH] This cell conditionally handles L1_claude. After Gemini
# data lands in verdict_scores_wide.csv, the same logic applies to L1_gemini.
# Manual extension may be needed — review the 'L1_claude' references below.
# Per-axis flip set vs anchor
def flip_set(wide, axis_cell, anchor=ANCHOR):
    common = wide[[anchor, axis_cell]].dropna()
    return set(common.index[np.sign(common[anchor]) != np.sign(common[axis_cell])])

axes = {'C': 'C2F1D1', 'F': 'C1F2D1', 'D': 'C1F1D2'}
if 'L1_claude' in wide.columns:
    axes['L1'] = 'L1_claude'
if 'L1_gemini' in wide.columns:  # [L1_GEMINI_PATCH]
    axes['L1_gemini'] = 'L1_gemini'

flip_sets = {ax: flip_set(wide, cell) for ax, cell in axes.items()}
n_total = len(wide)
marginals = {ax: len(fs) / n_total for ax, fs in flip_sets.items()}

rows = []
for a, b in combinations(axes.keys(), 2):
    fa, fb = flip_sets[a], flip_sets[b]
    obs_int = len(fa & fb)
    obs_un = len(fa | fb)
    obs_jac = obs_int / max(obs_un, 1)
    pa, pb = marginals[a], marginals[b]
    exp_int = pa * pb
    exp_un = pa + pb - exp_int
    exp_jac = exp_int / exp_un
    rows.append({
        'pair': f'{a}∩{b}',
        'obs_jaccard': round(obs_jac, 3),
        'chance_jaccard': round(exp_jac, 3),
        'obs_over_chance': round(obs_jac / exp_jac, 2) if exp_jac > 0 else None,
        'p_a': round(pa, 3),
        'p_b': round(pb, 3),
    })

jac_df = pd.DataFrame(rows)
print('Jaccard overlaps with chance baselines (under independence):')
print(jac_df.to_string(index=False))
jac_df.to_csv(results_dir / 'jaccard_with_baselines.csv', index=False)

# Conditional probability framing too (P(other | L1) under independence baseline = p_other)
if 'L1' in flip_sets:
    print('\nConditional P(X flips | L1 flips):')
    L1 = flip_sets['L1']
    for x in ['C', 'F', 'D']:
        obs_cond = len(L1 & flip_sets[x]) / max(len(L1), 1)
        chance_cond = marginals[x]
        print(f'  P({x} | L1): observed {obs_cond:.3f} vs chance {chance_cond:.3f}  (ratio {obs_cond/chance_cond:.2f}x)')

print('Saved: results/jaccard_with_baselines.csv')

Jaccard overlaps with chance baselines (under independence):
        pair  obs_jaccard  chance_jaccard  obs_over_chance   p_a   p_b
         C∩F        0.301           0.189             1.59 0.258 0.416
         C∩D        0.366           0.135             2.71 0.258 0.220
        C∩L1        0.274           0.173             1.58 0.258 0.346
 C∩L1_gemini        0.332           0.179             1.86 0.258 0.368
         F∩D        0.242           0.168             1.44 0.416 0.220
        F∩L1        0.305           0.233             1.31 0.416 0.346
 F∩L1_gemini        0.338           0.243             1.39 0.416 0.368
        D∩L1        0.335           0.155             2.16 0.220 0.346
 D∩L1_gemini        0.349           0.160             2.18 0.220 0.368
L1∩L1_gemini        0.417           0.217             1.92 0.346 0.368

Conditional P(X flips | L1 flips):
  P(C | L1): observed 0.376 vs chance 0.258  (ratio 1.46x)
  P(F | L1): observed 0.514 vs chance 0.416  (ratio 1.24x)
  P(

## Summary — robustness check headline numbers

End-of-notebook recap. All numbers below come from the cells above. Maps directly to verifier action items in `paper_claims.md` Pre-submission section.

In [8]:
summary = {
    'variance_matched_cot': {
        'mean_F1_full_delta_pp': float(vmc_df.full_delta_pp.mean()),
        'mean_F1_matched_delta_pp': float(vmc_df.matched_delta_pp.mean()),
        'F2_reference_delta_pp': float(f2_avg_delta),
        'mechanism_reading': (
            'variance_artefact_dominates' if vmc_df.matched_delta_pp.mean() < f2_avg_delta + 0.5
            else 'verbalisation_channel_survives' if vmc_df.matched_delta_pp.mean() > vmc_df.full_delta_pp.mean() - 0.5
            else 'intermediate'
        ),
    },
    'bootstrap_anchor_stability': {
        'n_comparisons': int(len(stability)),
        'n_tied_40_60_pct': int((stability['anchor_wins_pct'].between(40, 60)).sum()),
        'n_distinguishable': int(stability['distinguishable'].sum()),
        'cap_reference_anchor_wins_pct': 50.7,
    },
    'bootstrap_seed_sensitivity': {
        'B3_max_ci_spread_pp': float(seed_df[seed_df.bucket == 'B3_unresolved'].assign(
            spread=lambda d: d.ci_hi - d.ci_lo).spread.max()),
        'verdict': 'ROBUST' if seed_df.groupby('bucket').apply(
            lambda g: max(g.ci_lo.max() - g.ci_lo.min(), g.ci_hi.max() - g.ci_hi.min())
        ).max() < 2 else 'SENSITIVE',
    },
    'anchor_cell_ci': anchor_ci,
    'invalid_verdict_audit': {
        'total_calls': int(summary['total'].sum()) if 'total' in summary.columns else 0,
        'total_invalid': int(summary['invalid'].sum()) if 'invalid' in summary.columns else 0,
        'overall_invalid_pct': float(100 * summary['invalid'].sum() / summary['total'].sum()) if 'total' in summary.columns else 0,
    } if 'total' in summary.columns else 'see invalid_verdict_audit.csv',
    'jaccard_baselines': {
        'pair_with_max_obs_over_chance': str(jac_df.iloc[jac_df.obs_over_chance.idxmax()]['pair']),
        'max_obs_over_chance': float(jac_df.obs_over_chance.max()),
    },
}

with open(results_dir / 'robustness_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('=' * 60)
print('06d ROBUSTNESS CHECK SUMMARY')
print('=' * 60)
print(json.dumps(summary, indent=2, default=str))
print('\nSaved: results/robustness_summary.json')
print('\nAll 6 outputs:')
for f in ['variance_matched_cot.csv', 'bootstrap_anchor_stability.csv',
          'bootstrap_seed_sensitivity.csv', 'anchor_cell_ci.json',
          'invalid_verdict_audit.csv', 'jaccard_with_baselines.csv',
          'robustness_summary.json']:
    print(f'  results/{f}')

06d ROBUSTNESS CHECK SUMMARY
{
  "variance_matched_cot": {
    "mean_F1_full_delta_pp": 3.3274999999999997,
    "mean_F1_matched_delta_pp": 3.4425,
    "F2_reference_delta_pp": 0.4360766748557072,
    "mechanism_reading": "verbalisation_channel_survives"
  },
  "bootstrap_anchor_stability": {
    "n_comparisons": 9,
    "n_tied_40_60_pct": 2,
    "n_distinguishable": 5,
    "cap_reference_anchor_wins_pct": 50.7
  },
  "bootstrap_seed_sensitivity": {
    "B3_max_ci_spread_pp": 9.009999999999998,
    "verdict": "ROBUST"
  },
  "anchor_cell_ci": {
    "n": 496,
    "point_estimate_pct": 50.18,
    "bootstrap_mean_pct": 50.14763248062813,
    "ci_lo_pct": 46.503723694502305,
    "ci_hi_pct": 53.686931596110945,
    "arena_hard_published_pct": 48.6,
    "published_inside_ci": true,
    "gap_to_published_pp": 1.5799999999999983
  },
  "invalid_verdict_audit": {
    "total_calls": 9996,
    "total_invalid": 24,
    "overall_invalid_pct": 0.24009603841536614
  },
  "jaccard_baselines": {
    "